#### 1.Create and get widgets

In [0]:
sourceSystem= dbutils.widgets.get("sourceSystem").lower()

In [0]:
from pyspark.sql.functions import lower,trim,when,col,countDistinct
import datetime

####2.Calling params and utils notebook

In [0]:
%run ../utils/params

In [0]:
%run ../utils/utilsMethods

#### 3.Calling logging notebook 

In [0]:
%run ../utils/loggingNotebook

In [0]:
notebookPath = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
notebookName = notebookPath.split("/")[-1]
jobID = dbutils.notebook.entry_point.getDbutils().notebook().getContext().tags().get("jobId")
adbpipelineID=f"{sourceSystem}_{notebookName}_{jobID}"
properties={'custom_dimensions': {'notebookName':f'{notebookName}', 'jobID':f'{jobID}' ,'adbPipelineID':f'{adbpipelineID}'}}

####4.Reading config file and validating it

In [0]:
try:
    logger.info(f"Checking if config information file exists at the {jobConfigPath}",extra=properties)
        
    if not dbutils.fs.ls(jobConfigPath):
        raise Exception(f"Mandatory config information file is not found at {jobConfigPath}.")
    else:
        # Reading the configTable.csv file from the jobConfigPath
        logger.info("config information file exists in location", extra = properties)
        logger.info("Reading the config information file from given location", extra = properties)
        
        configDf = readSparkCsv(jobConfigPath)
        configDf = configDf.select([when(trim(lower(col(c)))=="null",None).otherwise(col(c)).alias(c) for c in configDf.columns])
        
        if configDf.isEmpty():
            raise ValueError("config information file is empty, no data present")
        else:
            logger.info("config information file has data and is NOT empty",extra=properties)      
            
except Exception as e:
    # Logging and raising the error message
    logger.exception(f"Error while loading data from config information file: {str(e)}", extra = properties)
    raise e

####5.Checking sourceSystem is valid or not 

In [0]:
# Check - If sourceSystem  are valid
try:
    # Checking if sourceSystem parameter is present in configDf
    if sourceSystem.lower() not in [row[0].lower() for row in (configDf.select("sourceSystem")).distinct().collect()]:
    
        raise ValueError(f"Invalid sourceSystem parameter: {sourceSystem} in the Workflow Parameter.Please provide a valid Source System Name.")
        
    logger.info(f"Parameter {sourceSystem} validated successfully from the Config File.", extra = properties)
        
except Exception as e:
    logger.exception(f"Invalid source System parameter: {sourceSystem} in the Workflow Parameter.Please provide a valid Source System Name :{str(e)}", extra = properties)
    raise e
   

#### 6. Filtering  the Config file  based on the Source System Name and Checking for  Not null records for the Mandatory Columns

In [0]:
try:
    configDfFiltered=configDf.filter(trim(lower(configDf.sourceSystem))==(sourceSystem).lower())
    missingCols = [col for col in preProcessMandatoryConfigCols if col not in configDfFiltered.columns]
    if missingCols:
        
        raise ValueError(f"Mandatory column(s) {missingCols} is/are missing from the config information file.")
    logger.info(f"Mandatory column(s) {preProcessMandatoryConfigCols} are present in config information file.",extra=properties)
 
    for column in preProcessMandatoryConfigCols:
        # Filtering the dataframe and creating a new df with records having null values

        nullCount = configDfFiltered.where(f"{column} is null").count()
        if nullCount > 0:
            # Log the error message
            logger.exception(f"Found {nullCount} null values in the column {column}.", extra=properties)
            raise ValueError(f"{nullCount} null values found in the mandatory column {column}. Job terminated.")

        logger.info(f"No null records found in {preProcessMandatoryConfigCols}", extra=properties)
        
    configDf = configDfFiltered.withColumn("activeFlag", trim(lower(configDfFiltered["activeFlag"])))\
                               .withColumn("sourceSystem",trim(lower(configDfFiltered["sourceSystem"])))\
                               .withColumn("workloadType",trim(lower(configDfFiltered["workloadType"])))

except exception as e:
    logger.exception(f"Error Occured while filtering the config file for {sourceSystem}: {str(e)}", extra = properties)
    raise e 


#### 7. jobControl table validation

In [0]:
# Reading jobControl Delta table into a dataframe

try:
    jobControlDf=spark.sql(f"select * from {jobControlTable}")
except Exception as e:
    logger.exception("An error occurred while loading the job control table:",extra=properties)
    raise Exception("An error occurred while loading the job control table: " + str(e))

try:

    dfColList = jobControlDf.columns

    if len(dfColList) != len(preProcessMandatoryJcCols):

        raise ValueError(f"Number of columns in jobControl table ({len(dfColList)}) does not match the expected number ({len(preProcessMandatoryJcCols)}).")
    
    # Check if all columns in preProcessMandatoryJcCols are present in the dataframe
    for col in preProcessMandatoryJcCols:
        if col not in dfColList:

            raise ValueError(f"Column '{col}' is missing in the jobControl table.")

    logger.info("Have Read the delta table into jobControlDf, and all mandatory columns are present in jobControl table",extra=properties)
    
except Exception as e:
    logger.exception(f"An error occurred while validating the columns of the jobControl table: {str(e)}",extra=properties)
    raise e
        

####8.Current Run updation 

In [0]:
from pyspark.sql.functions import col
import datetime
try:
    if configDf.filter(col('activeFlag')=='y').count()>0:
        if not jobControlDf.where(lower(col("sourceSystem")) == sourceSystem).count() > 0:
            if configDf.filter((col("workloadType")=='historical') | (col("workloadType")=='histincr')):
                InsertJcQuery = f"""INSERT INTO {jobControlTable} (sourceSystem, lastRunDate, currentRunDate) VALUES ('{sourceSystem}','1900-01-01 00:00:00','{datetime.datetime.now()}')"""
                spark.sql(InsertJcQuery)
            
                logger.info("Updation of currentRunDate with currenttimestamp and lastRunDate with null is done for historical load in jobControl table",extra=properties)
                
            elif configDf.filter(col("workloadType")=='incremental'):
                logger.exception("Check workloadType and enter them correctly",extra=properties)
                raise KeyError("Check the workloadType in configuration information file and enter them correctly")
                
        else: 
            if configDf.filter((col("workloadType")=='incremental') | (col("workloadType")=='histincr')).count()>0:
                spark.sql(f"Update {jobControlTable} SET currentRunDate ='{datetime.datetime.now()}' where sourceSystem='{sourceSystem}'")

                logger.info("Updation of currentRunDate with currenttimestamp is done for incremental load in jobControl")
                
    else:
        logger.info("No Records found with active flag Y in Job Configuration File",extra=properties)
        dbutils.notebook.exit(f"Notebook {notebookName} exited as the Active Flag is N ,Cannot proceed with the Loads")
        
except Exception as e:   
    logger.exception(f"Updation of currentRunDate with currenttimestamp in jobControl table has failed: {str(e)}",extra=properties)
    raise e                

In [0]:
dbutils.notebook.exit("Notebook ran successfully")